# KDE plot of parameter estimates

In [1]:
import os
import numpy as np
import pandas as pd
from fontTools.cffLib import width
from tqdm.notebook import tqdm
from shared.noise_color_plot import create_symetric_powerlaw_plot

In [2]:
COEF_VAR = 'b'
xaxisrange = [-2, 2]
width = 800
height = 400
dataset_type = 'xling-'

In [3]:
# DATA_PATH = '../eng/multiple-others-rescaled-tdelta/reports/all.parquet'
DATA_PATH = '../xling/main_process/reports/all.parquet'
NULL_PATH = '../xling/main_process/reports/null-all.parquet'

# NULL_PARAMS = '../../4-POWER-LAW-SCALING/reports/null.csv'

REPORTS_PATH = 'reports'
if not os.path.exists(REPORTS_PATH):
    os.mkdir(REPORTS_PATH)

DIGITAL_EXCLUSIVE_PATH = os.path.join(REPORTS_PATH, 'website')
if not os.path.exists(DIGITAL_EXCLUSIVE_PATH):
    os.mkdir(DIGITAL_EXCLUSIVE_PATH)

PUBLICATION_READY_PATH = os.path.join(REPORTS_PATH, 'pub')
if not os.path.exists(PUBLICATION_READY_PATH):
    os.mkdir(PUBLICATION_READY_PATH)

In [4]:
df = pd.read_parquet(DATA_PATH)
df.head()

,corpus,lang,cid,speaker,self,a,b,k
0,callfriend,eng,callfriend-eng_n-callfriend-eng_n-4175,callfriend-eng_n-callfriend-eng_n-4175-F1,True,0.055142,0.675193,149
1,callfriend,eng,callfriend-eng_n-callfriend-eng_n-4175,callfriend-eng_n-callfriend-eng_n-4175-F1,False,0.057633,0.717667,151
2,callfriend,eng,callfriend-eng_n-callfriend-eng_n-4175,callfriend-eng_n-callfriend-eng_n-4175-F1,False,0.091267,0.587841,68
3,callfriend,eng,callfriend-eng_n-callfriend-eng_n-4175,callfriend-eng_n-callfriend-eng_n-4175-F1,False,0.221269,0.698865,13
4,callfriend,eng,callfriend-eng_n-callfriend-eng_n-4175,callfriend-eng_n-callfriend-eng_n-4175-F2,False,0.127429,0.587873,51


In [5]:
dfn = pd.read_parquet(NULL_PATH)

In [6]:
# df['corpus'] = [c if 'callfriend' not in c else 'callfriend' for c in df['corpus']]
df['corpus'] = df['corpus'].replace({'grace': 'Miao-fNIRS'})

In [7]:
df['corpus'].value_counts()

corpus
callhome      3313
croation      2361
callfriend    1865
finchat        322
yiddish         30
Name: count, dtype: int64

In [8]:
# null_vals = pd.read_csv(NULL_PARAMS)
# null_vals = {se: v for se,v in null_vals[['self', 'b']].values}
# # null_vals

In [9]:
df.isna().sum()

corpus     0
lang       0
cid        0
speaker    0
self       0
a          0
b          0
k          0
dtype: int64

In [10]:
((df[COEF_VAR].values >= -2) & (df[COEF_VAR].values <= 2)).mean()

np.float64(0.9987327334938537)

## Visualizing results

In [11]:
# import plotly.graph_objects as go
# import plotly.figure_factory as ff

## All data

In [12]:
hist_data = [
    df[COEF_VAR].loc[df['self']].values,
    df[COEF_VAR].loc[~df['self']].values,
    dfn[COEF_VAR].values
]

groups = [
    'sp(x) = sp(y)',
    'sp(x) ≠ sp(y)',
    'null'
]

colors = [
    '#FFC30B',
    '#0041C2',
    '#404040'
]

In [13]:
fig1 = create_symetric_powerlaw_plot(
    data=hist_data,
    data_names=groups,
    data_color=colors,
    background_color_scale='blues',
    divisions_per_side_of_zero=100
)
fig1.show()

In [14]:
fig1.update_layout(
    margin=dict(l=0, r=0, t=30, b=0),
)
fig1.update_xaxes(automargin=True)
fig1.update_yaxes(automargin=True)
fig1.write_html(os.path.join(DIGITAL_EXCLUSIVE_PATH, dataset_type+'all-data.html'))
fig1.write_image(os.path.join(PUBLICATION_READY_PATH, dataset_type+'all-data.png'), scale=5, width=width, height=height)

### by corpus

In [15]:
corpus_col = 'lang'

In [16]:
figs = []

In [17]:
for corpus in df[corpus_col].unique():
    sub = df.loc[df[corpus_col].isin([corpus])]

    hist_data = [
        sub[COEF_VAR].loc[sub['self']].values,
        sub[COEF_VAR].loc[~sub['self']].values,
    ]

    groups = [
        'sp(x) = sp(y)',
        'sp(x) ≠ sp(y)',
    ]

    colors = [
        '#FFC30B',
        '#0041C2'
    ]

    figs += [
        create_symetric_powerlaw_plot(
            data=hist_data,
            data_names=groups,
            data_color=colors,
            background_color_scale='blues',
            divisions_per_side_of_zero=25,
            mean_indicator_line_width=5,
            show_legend=False
        )
    ]

In [18]:
for corpus, plot in list(zip(df[corpus_col].unique(), figs)):
    # print(corpus)
    # plot.show()
    plot.update_layout(
        showlegend=False,
        margin=dict(l=0, r=0, t=30, b=0),
    )
    plot.update_xaxes(automargin=True)
    plot.update_yaxes(automargin=True)
    plot.write_html(os.path.join(DIGITAL_EXCLUSIVE_PATH, dataset_type+corpus+'.html'))
    # plot.write_image(os.path.join(PUBLICATION_READY_PATH, 'xling-'+corpus+'.png'), scale=5, width=int(width/1.5))